# Notebook consacré à l'analyse des variables nominales à choix multiples

Exemple :


Avez-vous des productions déposées sur Octaviana, la bibliothèque numérique de l'université ? (Plusieurs réponses possibles)
* Des travaux universitaires
* Des publications
* Des formations
* Des captations d'évènements (colloque, journée d'étude)
* Je ne connais pas Octaviana


In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

In [ ]:
list_nominal_multiple = [x for x in df_col.label.loc[(df_col.type=="nominal_multiple")&(df_col.label!="q12_1_how_humanum")]]
list_nominal_multiple

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )
df0["q24_research_fields"]

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
df0.q3_octavi_depot

# Types de données

In [ ]:
split_multiple_choices(df0, column='q3_octavi_depot', index = "q45_clé")

In [ ]:
df0.loc[df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Non"
df0.loc[~df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Oui"
df_col.columns
new_variable = pd.DataFrame.from_dict(data={'name':['3. octavi_depot_rec'], 'label':['q3_octavi_rec'], 'group':['2_bib_num'], 'personal_data':False, 'type':['simple_nominal'], 'opened_question':[False],
       'type_panda':[df0.q3_octavi_rec.dtypes], 'no_question':['3'], 'question':[df_col.question.loc[df_col.label=="q3_octavi_depot"].iloc[0]]})
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(len(list_nominal_multiple[0:3]), figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple[0:3]):
    print(col)
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total_freq", y=col, data=gb_data,
                label="Non", color="b", ax=ax[n])
    sns.barplot(x="freq", y=col, data=gb_data,
                label="Oui", color="r", ax=ax[n])
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax[n].yaxis.set_label_text("")
    ax[n].xaxis.set_label_text("")
    ax[n].set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/multiple_choice_1.png", bbox_inches='tight', dpi = 200)

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(len(list_nominal_multiple[3:6]), figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple[3:6]):
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total", y=col, data=gb_data,
                label="Total", color="b", ax=ax[n])
    sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax[n].yaxis.set_label_text("")
    ax[n].xaxis.set_label_text("")
    ax[n].set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/multiple_choice_2.png", dpi = 200, bbox_inches='tight')

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(len(list_nominal_multiple[6:]), figsize=(10, 20))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple[6:]):
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total", y=col, data=gb_data,
                label="Total", color="b", ax=ax[n])
    sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])
    titre = df_col.question.loc[df_col.label==col].iloc[0]
    ax[n].yaxis.set_label_text("")
    ax[n].xaxis.set_label_text("")
    ax[n].set_title(titre.replace("<i>","(").replace("</i>",")"))
    

sns.despine(left=True, bottom=True)
plt.savefig(f"viz/multiple_choice_3.png", dpi = 200, bbox_inches='tight')

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])